# Presentation figures

Slide-ready figures for the main results. Reads the saved model outputs and fits nothing:

- `mixed_model.py` outputs (`fit_mixed_model.slurm`): the before/after and BACI effect per measure
- the primary distance run (`fit_distance_models.slurm`, `stage3/primary__auto/`): how effects change downstream
- the characteristic screen (`RUNS=modifiers`, `stage5/modifiers__auto/`): which site characteristics go with bigger or smaller changes
- the predictive feasibility outputs (`predictive_feasibility.slurm`, `data/processed/predictive/`): can a dam's change be predicted before removal?

Each figure is shown here and saved as a PNG (200 dpi) in `data/processed/presentation_figures/<case_study>/`.
Detailed diagnostics stay in `mixed_model_results.ipynb` and `distance_model_results.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
FIG_DIR = Path(os.environ.get("DAM_REMOVAL_FIG_DIR",
                           ROOT / "data" / "processed" / "presentation_figures" / CASE_STUDY))
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Label for the figure footers: which fit these results come from.
RUN_LABEL = "Final model fit, 2026-11-17 (corrected dam locations and station reaches)"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 12,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})

NAMES = {
    "water_temp": "Water temperature",
    "dissolved_oxygen": "Dissolved oxygen",
    "ph": "pH",
    "specific_conductance": "Specific conductance",
    "turbidity": "Turbidity",
    "phosphorus (Total)": "Total phosphorus",
    "orthophosphate (Dissolved)": "Dissolved orthophosphate",
    "tds": "Total dissolved solids",
    "ssc": "Suspended sediment (SSC)",
    "tss": "Total suspended solids",
}
# The measures shown in the headline figures, in this order.
CORE = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance",
        "turbidity", "phosphorus (Total)", "orthophosphate (Dissolved)"]

DESIGNS = {
    "before_after": ("Downstream, before vs. after", "#1f6f8b"),
    "baci": ("Downstream vs. upstream controls (BACI)", "#d4762c"),
}
MIN_DAMS_SOLID = 15     # below this, an estimate is labelled exploratory


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"Saved {FIG_DIR / name}.png")


def footer(fig, text, y=-0.02, width=None):
    import textwrap
    width = width or int(fig.get_figwidth() * 15)
    lines = [w for part in text.split("\n") for w in textwrap.wrap(part, width)]
    fig.text(0.01, y, "\n".join(lines), ha="left", va="top", fontsize=9.5, color="0.35")


def to_display(values, log_scale):
    """Model units -> slide units: % change for log10-scale measures."""
    values = np.asarray(values, dtype=float)
    return (10 ** values - 1) * 100 if log_scale else values


def unit_of(series, log_scale, unit=""):
    return "% change" if log_scale else {"deg C": "°C", "std units": "pH units"}.get(unit, unit)


def threshold_text(series):
    return rep.sesoi_label(series).replace("+/-", "±").replace("deg C", "°C").replace("std units", "pH units")

## 1. Headline: what changed after removal

One panel per measure, in its own units. **Dot** = average change across dams after removal;
**bar** = 95% confidence interval. **Filled** dots = the interval excludes zero (a detectable change);
**hollow** = can't distinguish from no change. Numbers in brackets = dams.

In [ ]:
results = pd.read_csv(MODEL_DIR / "results.csv")
results = results[results["error"].fillna("").eq("") & results["estimate"].notna()].copy()
log = results["log_scale"].astype(str).str.lower().eq("true")
results["est"] = np.where(log, results["pct_estimate"], results["estimate"])
results["lo"] = np.where(log, results["pct_ci_low"], results["ci_low"])
results["hi"] = np.where(log, results["pct_ci_high"], results["ci_high"])
results["log"] = log
results["detected"] = (results["lo"] > 0) | (results["hi"] < 0)


def headline(frame, series_list, title, name, ncols=4):
    series_list = [s for s in series_list if s in set(frame["series"])]
    nrows = int(np.ceil(len(series_list) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 2.5 * nrows), squeeze=False)
    for ax, series in zip(axes.flat, series_list):
        rows = frame[frame["series"] == series].set_index("model").reindex(list(DESIGNS)).dropna(subset=["est"])
        for i, (model, r) in enumerate(rows.iloc[::-1].iterrows()):
            label, color = DESIGNS[model]
            ax.plot([r["lo"], r["hi"]], [i, i], color=color, lw=2.5, solid_capstyle="round")
            ax.plot(r["est"], i, "o", ms=10, mec=color, mew=2,
                    mfc=color if r["detected"] else "white", zorder=3)
            tag = f"[{int(r['dams'])}]" + ("*" if r["dams"] < MIN_DAMS_SOLID else "")
            ax.annotate(tag, (r["hi"], i), xytext=(6, 0), textcoords="offset points",
                        va="center", fontsize=10, color="0.3")
        ax.axvline(0, color="0.2", lw=1)
        ax.set_yticks([])
        ax.set_ylim(-0.7, len(rows) - 0.3)
        r0 = rows.iloc[0]
        ax.set_title(NAMES.get(series, series), loc="left")
        ax.set_xlabel(unit_of(series, r0["log"], r0["unit"]))
        lo, hi = rows["lo"].min(), rows["hi"].max()
        pad = 0.25 * (hi - lo)
        ax.set_xlim(min(lo, 0) - pad * 0.3, max(hi, 0) + pad)
    for ax in list(axes.flat)[len(series_list):]:
        ax.set_visible(False)
    handles = [Line2D([], [], color=c, marker="o", ms=9, lw=2.5, label=l) for l, c in DESIGNS.values()]
    handles += [Line2D([], [], color="0.3", marker="o", ms=9, lw=0, label="Detectable change (filled)"),
                Line2D([], [], color="0.3", marker="o", ms=9, mfc="white", lw=0,
                       label="No detectable change (hollow)")]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.0 + 0.09 / nrows))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.0 + 0.2 / nrows)
    fig.tight_layout()
    footer(fig, "Average change in the 5 years after removal vs. the 5 years before, with 95% confidence "
                "intervals. Mixed-effects models (dam and station random effects; month, air temperature, rain).\n"
                f"[n] = dams; * = fewer than {MIN_DAMS_SOLID} dams (exploratory).   {RUN_LABEL}")
    save(fig, name)
    plt.show()


headline(results, CORE, "How water quality changed after dam removal", "01_headline_effects")

## 2. The two clearest results, in plain numbers

Before/after downstream, the design with the most dams: water was cooler and held more oxygen after removal.

In [ ]:
KEY = ["water_temp", "dissolved_oxygen"]
ba = results[(results["model"] == "before_after") & results["series"].isin(KEY)].set_index("series")

fig, axes = plt.subplots(1, len(KEY), figsize=(10, 3.4))
for ax, series in zip(np.atleast_1d(axes), KEY):
    if series not in ba.index:
        ax.set_visible(False); continue
    r = ba.loc[series]
    unit = unit_of(series, r["log"], r["unit"])
    color = "#2b7bba" if r["est"] < 0 else "#2a9d55"
    if not r["detected"]:
        color = "0.65"
    ax.barh([0], [r["est"]], height=0.5, color=color, alpha=0.85)
    ax.errorbar(r["est"], 0, xerr=[[r["est"] - r["lo"]], [r["hi"] - r["est"]]], fmt="none",
                ecolor="0.15", capsize=8, lw=2)
    ax.axvline(0, color="0.2", lw=1)
    ax.set_yticks([])
    ax.set_ylim(-0.6, 0.6)
    direction = ("warmer", "cooler") if series == "water_temp" else ("more oxygen", "less oxygen")
    word = (direction[0] if r["est"] > 0 else direction[1]) if r["detected"] else "no detectable change"
    ax.set_title(f"{NAMES[series]}: {r['est']:+.2f} {unit}\n", loc="left")
    ax.text(0, 1.02, f"{word}  ·  {int(r['dams'])} dams, {int(r['stations'])} stations",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=11, color="0.3")
    span = max(abs(r["lo"]), abs(r["hi"])) * 1.3
    ax.set_xlim(-span, span)
    ax.set_xlabel(f"Change after removal ({unit})")
found = [n for n, s in [("cooler", "water_temp"), ("more oxygenated", "dissolved_oxygen")]
         if s in ba.index and ba.loc[s, "detected"]
         and (ba.loc[s, "est"] < 0 if s == "water_temp" else ba.loc[s, "est"] > 0)]
title = (f"Downstream water became {' and '.join(found)} after removal" if found
         else "Temperature and oxygen after removal")
fig.suptitle(title, fontsize=15, fontweight="bold", y=1.08)
fig.tight_layout()
footer(fig, "Before/after downstream model; bar = average change, whiskers = 95% confidence interval.   " + RUN_LABEL,
       y=-0.04)
save(fig, "02_temperature_oxygen")
plt.show()

## 3. Does every dam agree?

Each dam's own estimated change (the average plus that dam's deviation). Most dams moving the same way
supports a general removal effect; a wide spread means dams differ.

In [ ]:
dam_effects = pd.read_csv(MODEL_DIR / "dam_effects.csv")

fig, axes = plt.subplots(1, len(KEY), figsize=(11, 3.8))
for ax, series in zip(np.atleast_1d(axes), KEY):
    d = dam_effects[(dam_effects["series"] == series) & (dam_effects["model"] == "before_after")]["dam_effect"]
    if d.empty:
        ax.set_visible(False); continue
    unit = unit_of(series, False, ba.loc[series, "unit"]) if series in ba.index else ""
    lo, hi = d.quantile([0.02, 0.98])
    shown = d.clip(lo, hi)
    better = (d < 0).mean() if series == "water_temp" else (d > 0).mean()   # share in the average's direction
    ax.hist(shown, bins=25, color="#1f6f8b", alpha=0.85, edgecolor="white")
    ax.axvline(0, color="0.2", lw=1.2)
    ax.axvline(d.mean(), color="#d4762c", lw=2, ls="--", label=f"average {d.mean():+.2f} {unit}")
    direction = "cooler" if series == "water_temp" else "more oxygen"
    ax.set_title(f"{NAMES[series]}: {better:.0%} of {len(d)} dams {direction}", loc="left")
    ax.set_xlabel(f"Dam's estimated change after removal ({unit})")
    ax.set_ylabel("Number of dams")
    ax.legend(frameon=False, loc="upper right")
fig.suptitle("Most dams moved in the same direction, but individual dams vary", fontsize=15,
             fontweight="bold", y=1.04)
fig.tight_layout()
footer(fig, "Dam-specific effects from the before/after mixed model (shrunk toward the average for dams with "
            "little data); the most extreme 2% on each side are drawn at the edge.   " + RUN_LABEL, y=-0.04)
save(fig, "03_dam_agreement")
plt.show()

## 4. How far downstream do the effects reach?

From the primary distance model: the estimated change at each distance below the dam (line) with its 95%
confidence band. **Dots** = the estimate at 1, 5, 10, 20 and 30 km. **Dashed lines** = the provisional
practical threshold (changes inside them are too small to matter). Where the shaded band covers zero, we can't tell.

In [ ]:
SERIES_BY_DISTANCE = ["water_temp", "dissolved_oxygen", "ph"]   # edit to show other measures

stage3_results, stage3_effects, _ = rep.load_runs(MODEL_DIR / "stage3")
primary = stage3_results[(stage3_results["run"] == rep.PRIMARY[0]) & (stage3_results["re"] == rep.PRIMARY[1])]
curve = rep.effects_curve(primary)
points = rep.effects_by_distance(stage3_effects)
points = points[points["primary"] & points["distance_km"].notna()]
SPEC_TO_DESIGN = {"before/after": "before_after", "BACI": "baci"}


def distance_figure(series):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
    log_scale = series.split(" (")[0] in mm.SETTINGS["log_parameters"]
    unit = unit_of(series, log_scale, curve.loc[curve["series"] == series, "unit"].iloc[0]
                   if (curve["series"] == series).any() else "")
    sesoi = rep.SESOI.get(series)
    for ax, design in zip(axes, ["before/after", "BACI"]):
        label, color = DESIGNS[SPEC_TO_DESIGN[design]]
        c = curve[(curve["series"] == series) & (curve["design"] == design)]
        p = points[(points["series"] == series) & (points["design"] == design)]
        ax.axhline(0, color="0.2", lw=1)
        if sesoi is not None:
            band = to_display([-sesoi, sesoi], log_scale)
            for i, b in enumerate(band):
                ax.axhline(b, color="0.45", ls="--", lw=1.2,
                           label=f"Practical threshold ({threshold_text(series)})" if i == 0 else None)
        if c.empty:
            ax.text(0.5, 0.5, "Not estimated (too few dams)", transform=ax.transAxes, ha="center",
                    color="0.4")
        else:
            x = c["distance_km"]
            ax.fill_between(x, to_display(c["ci_low"], log_scale), to_display(c["ci_high"], log_scale),
                            color=color, alpha=0.2, lw=0, label="95% confidence band")
            ax.plot(x, to_display(c["estimate"], log_scale), color=color, lw=2.5, label="Estimated change")
            if len(p):
                est = to_display(p["estimate"], log_scale)
                ax.errorbar(p["distance_km"], est,
                            yerr=[est - to_display(p["ci_low"], log_scale),
                                  to_display(p["ci_high"], log_scale) - est],
                            fmt="o", color=color, capsize=4, ms=6)
            dams = int(c["dams"].iloc[0])
            label += f"  ({dams} dams{'; exploratory' if dams < MIN_DAMS_SOLID else ''})"
        ax.set_xscale("log")
        ax.set_xticks([1, 2, 5, 10, 20, 30], ["1", "2", "5", "10", "20", "30"])
        ax.set_xlabel("Distance downstream of the removed dam (km, log scale)")
        ax.set_title(label, loc="left", fontsize=12)
    axes[0].set_ylabel(f"Change after removal ({unit})")
    axes[0].legend(frameon=False, loc="best", fontsize=10)
    fig.suptitle(f"{NAMES.get(series, series)}: change after removal by distance downstream",
                 fontsize=15, fontweight="bold", y=1.05)
    fig.tight_layout()
    footer(fig, "Distance mixed-effects model (stations within 30 km; adjusted for dilution, season and "
                "weather). Threshold is a provisional analyst choice, not a regulatory standard.\n" + RUN_LABEL,
           y=-0.05)
    save(fig, f"04_distance_{rep.slug(series)}")
    plt.show()


for series in SERIES_BY_DISTANCE:
    distance_figure(series)

## 5. How much data each result rests on

Dams per measure in each design. BACI (with upstream controls) is the stronger design but has far fewer dams,
which is why several BACI intervals are wide.

In [ ]:
all_rows = pd.read_csv(MODEL_DIR / "results.csv")
counts = (all_rows[all_rows["series"].isin(NAMES)]
          .pivot_table(index="series", columns="model", values="dams", aggfunc="first")
          .reindex(columns=list(DESIGNS)).fillna(0))
counts = counts.loc[counts.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 0.55 * len(counts) + 1.2))
y = np.arange(len(counts))
for offset, (model, (label, color)) in zip([0.2, -0.2], DESIGNS.items()):
    ax.barh(y + offset, counts[model], height=0.38, color=color, label=label)
    for yi, v in zip(y + offset, counts[model]):
        ax.text(v + 1.5, yi, f"{int(v)}", va="center", fontsize=10, color="0.25",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.5))
ax.axvline(MIN_DAMS_SOLID, color="0.4", ls=":", lw=1.5, zorder=0)
ax.text(MIN_DAMS_SOLID, len(counts) - 0.4, f" {MIN_DAMS_SOLID} dams", color="0.4", fontsize=10, va="bottom")
ax.set_yticks(y, [NAMES[s] for s in counts.index])
ax.set_xlabel("Number of removed dams in the model")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Sample size: dams per measure and design", loc="left", fontsize=15)
fig.tight_layout()
footer(fig, "Dams with enough monitoring before and after removal (≥3 samples in ≥2 years on each side), "
            "after model filters. Below 15 dams, results are exploratory.   " + RUN_LABEL, y=-0.03)
save(fig, "05_sample_size")
plt.show()

## 6. Robustness: did the location corrections change the answer?

The same models before the manual review (frozen 2026-11-10) and after correcting dam locations and station
reaches. Overlapping estimates = the conclusions don't depend on those corrections.

In [ ]:
FROZEN = Path(os.environ.get("DAM_REMOVAL_FROZEN_DIR",
                          ROOT / "data" / "processed" / "frozen" / "models-frozen-2026-11-10"))
frozen_path = next((p for p in sorted(FROZEN.rglob("results.csv")) if (p.parent / "dam_effects.csv").exists()),
                   None) if FROZEN.exists() else None

if frozen_path is None:
    print(f"No frozen mixed-model results under {FROZEN}; skipping.")
else:
    before = pd.read_csv(frozen_path)
    before = before[before["error"].fillna("").eq("") & before["estimate"].notna()].copy()
    blog = before["log_scale"].astype(str).str.lower().eq("true")
    before["est"] = np.where(blog, before["pct_estimate"], before["estimate"])
    before["lo"] = np.where(blog, before["pct_ci_low"], before["ci_low"])
    before["hi"] = np.where(blog, before["pct_ci_high"], before["ci_high"])
    both = results.merge(before[["series", "model", "est", "lo", "hi", "dams"]], on=["series", "model"],
                         suffixes=("", "_before"))
    shown = [s for s in CORE if s in set(both["series"])]
    fig, axes = plt.subplots(2, len(shown), figsize=(2.05 * len(shown), 6.2), squeeze=False)
    for row, model in enumerate(DESIGNS):
        label, color = DESIGNS[model]
        for col, series in enumerate(shown):
            ax = axes[row, col]
            r = both[(both["series"] == series) & (both["model"] == model)]
            if r.empty:
                ax.axis("off")
                ax.text(0.5, 0.5, "too few\ndams", ha="center", va="center", color="0.5",
                        transform=ax.transAxes, fontsize=10)
                continue
            r = r.iloc[0]
            for x, (e, l, h, filled) in enumerate([(r["est_before"], r["lo_before"], r["hi_before"], False),
                                                   (r["est"], r["lo"], r["hi"], True)]):
                ax.plot([x, x], [l, h], color=color, lw=2.2)
                ax.plot(x, e, "o", ms=8, mec=color, mew=2, mfc=color if filled else "white")
            ax.axhline(0, color="0.2", lw=0.8)
            ax.set_xlim(-0.6, 1.6)
            ax.set_xticks([0, 1], [f"before\n({int(r['dams_before'])})", f"after\n({int(r['dams'])})"],
                          fontsize=9)
            ax.tick_params(axis="y", labelsize=9)
            if row == 0:
                unit = unit_of(series, r["log"], r["unit"])
                ax.set_title(f"{NAMES[series]}\n({unit})", fontsize=10)
        axes[row, 0].annotate(label.replace(" (BACI)", "\n(BACI)").replace(", before", ",\nbefore"),
                              (-0.75, 0.5), xycoords="axes fraction", rotation=90, ha="center",
                              va="center", fontsize=10.5, fontweight="bold", color=color)
    fig.suptitle("Results before vs. after correcting dam and station locations", fontsize=15,
                 fontweight="bold", y=1.03)
    fig.tight_layout()
    footer(fig, "Hollow = frozen fit before the manual review (2026-11-10); filled = final fit. Bars = 95% "
                "confidence intervals; (n) = dams.\n181 dam locations corrected; all 723 analysis stations "
                "checked by hand, 24 reach assignments corrected.", y=-0.01)
    save(fig, "06_robustness_corrections")
    plt.show()

## 7. Which site characteristics go with bigger or smaller changes?

Each characteristic was tested alone: is the change after removal larger at dams that are taller, have bigger
rivers, or have more forest, farmland or development upstream? (30 km downstream, adjusted for distance and
dilution.) Many measures × characteristics were tested, so only results that survive a multiple-testing
correction (**q < 0.1**) count as findings. These are associations across dams, not causes.

In [ ]:
STAGE5 = MODEL_DIR / "stage5"


def stage5(run, name):
    """stage5/<run>__<random effects>/<name>, preferring the reported 'auto' runs."""
    found = {p.name.split("__")[-1]: p for p in STAGE5.glob(f"{run}__*")}
    folder = next((found[r] for r in ("auto", "slopes", "intercepts") if r in found), None)
    path = folder / name if folder else None
    return pd.read_csv(path) if path is not None and path.exists() and path.stat().st_size > 1 else pd.DataFrame()


screen = stage5("modifiers", "modifier_screen.csv")
mod_at = stage5("modifiers", "effects_at.csv")
mod_corr = stage5("audit_30km", "modifier_correlations.csv")
if mod_corr.empty:
    mod_corr = stage5("modifiers", "modifier_correlations.csv")

# Characteristics shown (distance is in section 4), with slide labels.
TRAITS = {"dam_height": "Dam height", "reservoir_volume": "Reservoir size\n(index)",
          "reservoir_area": "Mapped\nreservoir area", "dam_drainage_area": "River size\n(drainage area)",
          "pct_forest": "% forest\nupstream", "pct_agriculture": "% farmland\nupstream",
          "pct_developed": "% developed\nupstream"}
Q_MAX = 0.10
SHORT = {"baci": "BACI", "before_after": "Before/after"}


def trait_value(name, v):
    """A characteristic's value in plain units, for tick labels and tables."""
    if pd.isna(v):
        return ""
    if name.startswith("pct_"):
        return f"{v * 100 if v <= 1 else v:.0f}%"
    return {"dam_height": f"{v:.0f} ft", "dam_drainage_area": f"{v:,.0f} km²",
            "reservoir_area": f"{v:.2g} km²"}.get(name, f"{v:.2g}")


def plain_unit(series):
    """Slide unit: % change for log10 measures, else the threshold's unit (°C, mg/L, pH units)."""
    if series.split(" (")[0] in mm.SETTINGS["log_parameters"]:
        return "% change"
    return threshold_text(series).lstrip("±0123456789. ")


if screen.empty:
    print("No characteristic screen yet: RUNS=modifiers sbatch data/slurm/fit_distance_models.slurm")
    traits = screen
else:
    traits = screen[screen["error"].fillna("").eq("") & screen["series"].isin(CORE)
                    & screen["characteristic"].isin(TRAITS)].copy()
    traits["per_thr"] = traits["modifier_slope"] / traits["series"].map(rep.SESOI)
    traits["ci_excludes_0"] = (traits["modifier_ci_low"] > 0) | (traits["modifier_ci_high"] < 0)
    traits["finding"] = traits["ci_excludes_0"] & (traits["q_value"] < Q_MAX)
    for design, label in SHORT.items():
        n = int(traits.loc[traits["design"].eq(design), "finding"].sum())
        print(f"{label}: {n} measure × characteristic pair(s) with q < {Q_MAX}")

### 7a. Overview: every measure × characteristic tested

**Large filled dot** = a finding (q < 0.1); its size = how strong the link is. **Small hollow dot** = a hint
(95% interval excludes 0) that doesn't survive the multiple-testing correction. **Grey dot** = no clear link.
Orange = the change is more **positive** (bigger rise or smaller drop) at dams with more of the characteristic;
blue = more **negative**.

In [ ]:
UP, DOWN = "#d4762c", "#1f6f8b"

if len(traits):
    designs = [d for d in SHORT if d in set(traits["design"])]
    rows = [s for s in CORE if s in set(traits["series"])]
    fig, axes = plt.subplots(1, len(designs), figsize=(7.2 * len(designs), 0.62 * len(rows) + 2.2),
                             squeeze=False, sharey=True)
    for ax, design in zip(axes[0], designs):
        g = traits[traits["design"] == design].set_index(["series", "characteristic"])
        cols = [t for t in TRAITS if t in set(g.index.get_level_values("characteristic"))]
        for i, s in enumerate(rows):
            for j, t in enumerate(cols):
                if (s, t) not in g.index:
                    continue
                r = g.loc[(s, t)]
                color = UP if r["per_thr"] > 0 else DOWN
                if r["finding"]:
                    size = 120 + 380 * min(abs(r["per_thr"]), 1.5) / 1.5
                    ax.scatter(j, i, s=size, color=color, zorder=3)
                    ax.annotate(f"{int(r['dams'])} dams", (j, i), xytext=(0, -17), textcoords="offset points",
                                ha="center", va="top", fontsize=8.5, color="0.3")
                elif r["ci_excludes_0"]:
                    ax.scatter(j, i, s=70, facecolor="white", edgecolor=color, lw=1.8, zorder=3)
                else:
                    ax.scatter(j, i, s=14, color="0.75", zorder=3)
        ax.set_xticks(range(len(cols)), [TRAITS[c].replace("\n", " ") for c in cols], fontsize=10.5,
                      rotation=35, ha="right", rotation_mode="anchor")
        ax.set_yticks(range(len(rows)), [NAMES[s] for s in rows])
        ax.set_xlim(-0.6, len(cols) - 0.4)
        ax.set_ylim(len(rows) - 0.4, -0.6)
        ax.grid(color="0.93", lw=0.8)
        ax.set_axisbelow(True)
        ax.spines[["left", "bottom"]].set_visible(False)
        ax.tick_params(length=0)
        ax.set_title(DESIGNS[design][0], loc="left", color=DESIGNS[design][1])
    handles = [Line2D([], [], marker="o", lw=0, ms=12, color=UP, label="Finding: change more positive with more"),
               Line2D([], [], marker="o", lw=0, ms=12, color=DOWN, label="Finding: change more negative with more"),
               Line2D([], [], marker="o", lw=0, ms=8, mfc="white", mec="0.4", mew=1.8,
                      label="Hint only (not significant after correction)"),
               Line2D([], [], marker="o", lw=0, ms=4, color="0.75", label="No clear link")]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.07))
    fig.suptitle("Which site characteristics go with bigger or smaller changes after removal?",
                 fontsize=15, fontweight="bold", y=1.13)
    fig.tight_layout()
    footer(fig, "Each characteristic tested alone in the distance mixed model (stations within 30 km; adjusted for "
                f"distance, dilution, season and weather). Finding = Benjamini–Hochberg q < {Q_MAX} across all "
                "measures × characteristics of a design. Dot size = change per standard deviation of the "
                "characteristic, relative to the practical threshold. Dams missing a characteristic are left out "
                "of that test. Associations, not causes.\n" + RUN_LABEL, y=-0.02)
    save(fig, "07_characteristics_overview")
    plt.show()

### 7b. The findings, in plain numbers

For each finding: the estimated change after removal at a **typical dam (50th percentile)** and at dams low
(10th) and high (90th) in that characteristic. **Grey band** = too small to matter (practical threshold).
Fewer than 15 dams = exploratory.

In [ ]:
MAX_PANELS = 4

found = (traits[traits["finding"]].assign(order=lambda f: f["design"].map({"baci": 0, "before_after": 1}))
         .sort_values(["order", "q_value"]).head(MAX_PANELS)) if len(traits) else traits
if found.empty:
    print(f"No characteristic passes q < {Q_MAX}; nothing to show.")
elif mod_at.empty:
    print("No effects_at.csv in the modifiers run.")
else:
    pts = mod_at[mod_at["characteristic_percentile"].notna()]
    fig, axes = plt.subplots(1, len(found), figsize=(4.3 * len(found), 4.3), squeeze=False)
    for ax, (_, r) in zip(axes[0], found.iterrows()):
        s, t = r["series"], r["characteristic"]
        log_scale = s.split(" (")[0] in mm.SETTINGS["log_parameters"]
        g = pts[(pts["series"] == s) & (pts["spec"] == r["spec"])].sort_values("characteristic_percentile")
        est, lo, hi = (to_display(g[c], log_scale) for c in ("estimate", "ci_low", "ci_high"))
        color = DESIGNS[r["design"]][1]
        band = to_display([-rep.SESOI[s], rep.SESOI[s]], log_scale)
        ax.axhspan(band[0], band[1], color="0.9", lw=0, label=f"Too small to matter ({threshold_text(s)})")
        ax.axhline(0, color="0.2", lw=1)
        x = np.arange(len(g))
        ax.errorbar(x, est, yerr=[est - lo, hi - est], fmt="o-", color=color, ms=9, lw=2, capsize=5)
        ax.set_xticks(x, [f"{int(p)}th pct\n{trait_value(t, v)}"
                          for p, v in zip(g["characteristic_percentile"], g["characteristic_value"])])
        ax.set_xlim(-0.5, len(g) - 0.5)
        ax.set_ylabel(f"Change after removal ({plain_unit(s)})")
        ax.set_xlabel(TRAITS[t].replace("\n", " "))
        tag = "  · exploratory" if r["dams"] < MIN_DAMS_SOLID else ""
        ax.set_title(f"{NAMES[s]}\nby {TRAITS[t].replace(chr(10), ' ').lower()}", loc="left", pad=22)
        ax.text(0, 1.01, f"{SHORT[r['design']]} · {int(r['dams'])} dams · q = {r['q_value']:.2f}{tag}",
                transform=ax.transAxes, fontsize=9.5, color="0.35", va="bottom")
    axes[0][0].legend(frameon=False, loc="upper left", fontsize=9)
    # Characteristics that travel together for the same measure (likely one pattern, not two).
    notes = []
    if len(mod_corr) and {"a", "b", "spearman_rho"} <= set(mod_corr):
        for (s, d), grp in found.groupby(["series", "design"]):
            ts = list(grp["characteristic"])
            for i in range(len(ts)):
                for j in range(i + 1, len(ts)):
                    c = mod_corr[(mod_corr.get("design", d) == d)
                                 & (((mod_corr["a"] == ts[i]) & (mod_corr["b"] == ts[j]))
                                    | ((mod_corr["a"] == ts[j]) & (mod_corr["b"] == ts[i])))]
                    if len(c) and abs(c["spearman_rho"].iloc[0]) >= 0.5:
                        notes.append(f"{NAMES[s]}: {TRAITS[ts[i]].replace(chr(10), ' ')} and "
                                     f"{TRAITS[ts[j]].replace(chr(10), ' ')} move together across these dams "
                                     f"(ρ = {c['spearman_rho'].iloc[0]:+.1f}), so they are probably one pattern.")
    fig.suptitle("Where the change after removal depends on the site", fontsize=15, fontweight="bold", y=1.1)
    fig.tight_layout()
    footer(fig, "Estimated change at dams at the 10th, 50th and 90th percentile of the characteristic (others at "
                "their average), with 95% confidence intervals; distance mixed model, 30 km, adjusted for distance "
                "and dilution. " + " ".join(notes) + "\n" + RUN_LABEL, y=-0.03)
    save(fig, "08_characteristic_findings")
    plt.show()

### 7c. Summary table

In [ ]:
if len(traits) and len(mod_at):
    pts = mod_at[mod_at["characteristic_percentile"].isin([10, 90])]
    rows_out = []
    for _, r in traits[traits["finding"]].sort_values(["design", "q_value"]).iterrows():
        s = r["series"]
        log_scale = s.split(" (")[0] in mm.SETTINGS["log_parameters"]
        g = pts[(pts["series"] == s) & (pts["spec"] == r["spec"])].set_index("characteristic_percentile")
        if not {10, 90} <= set(g.index):
            continue
        lo_v, hi_v = to_display(g.loc[[10, 90], "estimate"], log_scale)
        unit = "%" if log_scale else " " + plain_unit(s)
        rows_out.append({
            "Measure": NAMES[s], "Design": SHORT[r["design"]],
            "Characteristic": TRAITS[r["characteristic"]].replace("\n", " "),
            "Change: low → high dams": f"{lo_v:+.2g}{unit} → {hi_v:+.2g}{unit}",
            "Dams": int(r["dams"]), "q": round(r["q_value"], 2)})
    table = pd.DataFrame(rows_out)
    if table.empty:
        print(f"No characteristic passes q < {Q_MAX}.")
    else:
        display(table.style.hide(axis="index"))
    print("Low / high = dams at the 10th / 90th percentile of the characteristic. "
          f"Fewer than {MIN_DAMS_SOLID} dams = exploratory.")

**How to say it on a slide**
- Most characteristics show **no clear link**: for most measures, the change after removal looks similar at big
  and small dams and across land uses.
- The few findings come from the BACI design with only about 10–15 dams each, so they are **leads for future work**, not
  rules. They held up when distance and dilution were not adjusted for.
- "Reservoir size" is a constructed index (height, slope, river size), not a measured sediment volume.

## 8. Can we predict which removals will change water quality the most?

Section 3 showed that dams differ. Here's the follow-up question: using only things known **before** a removal
(dam height, river size, upstream land use, climate, dam age, reservoir area), can we predict how much a dam's
downstream water quality will change?

How it was tested: each dam gets one number, its own change (stations within 10 km, years 1–5 after removal).
A model learns from dams in all regions but one, predicts the dams in the held-out region, and this repeats for
every region. That mimics the real question: "a removal somewhere new". The rules for what counts as
"predictable" were written down before any model was fit (`docs/predictive_feasibility_prespec.md`).
Full diagnostics are in `predictive_feasibility.ipynb`.

In [ ]:
from analysis import predictive_feasibility as pf

PRED_DIR = Path(os.environ.get("DAM_REMOVAL_PREDICTIVE_DIR", pf.output_dir_for(CASE_STUDY, ROOT)))
PRED_LABEL = "Predictive feasibility analysis, 2026-11-19 (dam targets built from the models frozen 2026-11-18)"
PRED_SERIES = list(pf.TARGET_SERIES)
REAL, PRED = "#1f6f8b", "#d4762c"


def pred_csv(name):
    path = PRED_DIR / f"{name}.csv"
    if not path.exists() or path.stat().st_size <= 1:
        return pd.DataFrame()
    return pd.read_csv(path, dtype={"AR_ID": str, "huc2": str, "fold": str})


def is_log(series):
    return series.split(" (")[0] in mm.SETTINGS["log_parameters"]


def is_true(col):
    return col.astype(str).str.lower().eq("true")


cv_metrics, cv_dec, cv_pred = pred_csv("cv_metrics"), pred_csv("cv_decisions"), pred_csv("cv_predictions")
reliab = pred_csv("targets_reliability")
if cv_metrics.empty or reliab.empty:
    print(f"No predictive outputs in {PRED_DIR}: run STAGE=audit, targets and cv "
          "(data/slurm/predictive_feasibility.slurm)")
else:
    # Share of the dam-to-dam spread that is real: median and range over the four estimates.
    rel = (reliab[reliab["version"].eq("primary_10km") & reliab["se_kind"].eq("se_total")]
           .groupby("series")["reliability"].agg(["median", "min", "max"]).reindex(PRED_SERIES))
    loro = cv_metrics[cv_metrics["analysis"].eq("primary") & cv_metrics["scheme"].eq("loro")]
    ridge = loro[loro["model"].eq("C")].set_index("series").reindex(PRED_SERIES)
    mean_only = loro[loro["model"].eq("A")].set_index("series").reindex(PRED_SERIES)
    go = cv_dec[cv_dec["model"].eq("C")].assign(go=lambda d: is_true(d["go"]))
    print(f"Measures predictable (primary analysis): {int(go.loc[go['analysis'].eq('primary'), 'go'].sum())} of "
          f"{len(PRED_SERIES)};  sensitivity analyses passing: "
          f"{int(go.loc[~go['analysis'].eq('primary'), 'go'].sum())} of {int((~go['analysis'].eq('primary')).sum())}")

### 8a. The headline: real differences, but not predictable

**Left:** how much of the dam-to-dam spread is real rather than measurement noise. That is roughly the most a
perfect model could ever explain. **Right:** how much the model actually explained for dams in regions it
had never seen. Both are on the same 0–100% scale. The dashed line (5%) is the pre-set bar for "predictable".

In [ ]:
if len(cv_metrics) and len(reliab):
    y = np.arange(len(PRED_SERIES))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True, gridspec_kw={"wspace": 0.08})

    med = rel["median"] * 100
    a1.barh(y, med, height=0.6, color=REAL, alpha=0.9)
    a1.errorbar(med, y, xerr=[med - rel["min"] * 100, rel["max"] * 100 - med], fmt="none", ecolor="0.25",
                capsize=4, lw=1.5)
    for yi, v, hi in zip(y, med, rel["max"] * 100):
        a1.text(hi + 2, yi, f"{v:.0f}%", va="center", fontsize=11, color="0.2")
    a1.set_xlim(0, 100)
    a1.set_xlabel("% of the dam-to-dam spread that is real")
    a1.set_title("Dams really do differ…", loc="left", color=REAL)
    a1.set_yticks(y, [NAMES[s] for s in PRED_SERIES])
    a1.set_ylim(len(y) - 0.5, -1.3)          # inverted, with headroom for the labels on the right

    r2, lo, hi = (ridge[c] * 100 for c in ("r2", "r2_boot_q05", "r2_boot_q95"))
    left = min(-15, np.floor(lo.min() / 5) * 5 - 2)
    a2.axvspan(left, 0, color="0.94", lw=0)
    a2.text(left / 2, -0.85, "worse than\naverage", fontsize=9, color="0.45", ha="center",
            va="center")
    a2.axvline(0, color="0.2", lw=1)
    a2.axvline(pf.R2_MIN * 100, color="#c0392b", ls="--", lw=1.5)
    a2.text(pf.R2_MIN * 100 + 1.5, -0.85, "bar for 'predictable' (5%)", color="#c0392b", fontsize=10,
            va="center")
    a2.errorbar(r2, y, xerr=[r2 - lo, hi - r2], fmt="o", color=PRED, ms=9, capsize=4, lw=2)
    for yi, v, h in zip(y, r2, hi):
        a2.text(max(h, pf.R2_MIN * 100) + 3, yi, f"{v:+.1f}%", va="center", fontsize=11, color="0.2")
    a2.set_xlim(left, 100)
    a2.set_xlabel("% of the spread predicted for dams in unseen regions")
    a2.set_title("…but dam traits don't predict how much", loc="left", color=PRED)
    a2.tick_params(axis="y", length=0)

    fig.suptitle("Removals differ in their effects, but we can't predict which ones will change most",
                 fontsize=15, fontweight="bold", y=1.03)
    footer(fig, "Left: reliability of each dam's estimated change (between-dam variance ÷ (between-dam variance + "
                "measurement error)); bar = median of four estimates, whiskers = their range. Right: out-of-sample "
                "R² of a ridge regression on dam height, drainage area, upstream land use, pre-removal climate, dam "
                "age and reservoir area, holding out one HUC2 region at a time; whiskers = 90% bootstrap range over "
                "regions. A random forest did no better.\n" + PRED_LABEL, y=-0.03)
    save(fig, "09_predictability")
    plt.show()

### 8b. What "not predictable" looks like

Each dot is one dam in a region the model didn't see: its **predicted** change (across) vs. its **actual** change
(up). A model with skill would put the dots along the diagonal. Here the predictions bunch near the average while
the actual changes spread far above and below. That vertical cloud means the predictions carry almost no
information.

The dots fall in a few vertical columns, one per held-out region. That's expected: the model chose heavy shrinkage
because the traits didn't help, so for every dam in a region it predicts roughly the average of the other regions'
dams.

In [ ]:
if len(cv_pred):
    p = cv_pred[cv_pred["analysis"].eq("primary")]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for ax, s in zip(axes.flat, PRED_SERIES):
        g = p[p["series"].eq(s)]
        log_scale = is_log(s)
        obs, prd = to_display(g["y"], log_scale), to_display(g["pred_C"], log_scale)
        lim = np.nanpercentile(np.r_[obs, prd], [1, 99])
        lim = lim + np.array([-1, 1]) * 0.08 * (lim[1] - lim[0])
        ax.plot(lim, lim, color="0.6", lw=1.2, ls="--", label="perfect prediction")
        ax.axhline(0, color="0.2", lw=0.6)
        ax.axvline(0, color="0.2", lw=0.6)
        ax.scatter(np.clip(prd, *lim), np.clip(obs, *lim), s=22, color=PRED, alpha=0.65, edgecolor="white", lw=0.4)
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_aspect("equal", adjustable="box")
        unit = "% change" if log_scale else threshold_text(s).lstrip("±0123456789. ")
        ax.set_title(f"{NAMES[s]}  (R² = {ridge.loc[s, 'r2']:+.2f})", loc="left", fontsize=12)
        ax.set_xlabel(f"Predicted ({unit})", fontsize=10.5)
        ax.set_ylabel(f"Actual ({unit})", fontsize=10.5)
        ax.text(0.03, 0.97, f"{len(g)} dams", transform=ax.transAxes, va="top", fontsize=10, color="0.35")
    axes.flat[0].legend(frameon=False, loc="lower right", fontsize=9.5)
    fig.suptitle("Predicted vs. actual change for dams in regions the model never saw", fontsize=15,
                 fontweight="bold", y=1.0)
    fig.tight_layout()
    footer(fig, "Each dam's actual change (stations within 10 km, years 1–5 after removal, adjusted for season and "
                "weather, standardized to 5 km downstream) vs. the ridge model's prediction made with that dam's "
                "whole region held out. Extreme dams (outside the 1st–99th percentile) are drawn at the edge. Dots stack in columns, one "
                "per held-out region: the model found nothing worth using, so it shrank to the training average.\n"
                + PRED_LABEL, y=-0.01)
    save(fig, "10_predicted_vs_actual")
    plt.show()

### 8c. Summary table

In [ ]:
if len(cv_metrics) and len(reliab):
    def nat(v):
        return str(v).replace("deg C", "°C").replace("std units", "pH units")

    sens = go[~go["analysis"].eq("primary")].groupby("series")["go"].agg(["sum", "size"])
    sizes = cv_metrics[cv_metrics["analysis"].eq("primary") & cv_metrics["scheme"].eq("loro")
                       & cv_metrics["model"].eq("C")].set_index("series")["dams"]
    prim_go = go[go["analysis"].eq("primary")].set_index("series")["go"]
    rows_out = []
    for s in PRED_SERIES:
        rows_out.append({
            "Measure": NAMES[s],
            "Dams": int(sizes.get(s, 0)),
            "Real dam-to-dam differences": f"{rel.loc[s, 'median']:.0%}",
            "Predicted for unseen regions (R², 90% range)":
                f"{ridge.loc[s, 'r2']:+.1%}  ({ridge.loc[s, 'r2_boot_q05']:+.0%} to {ridge.loc[s, 'r2_boot_q95']:+.0%})",
            "Typical error: model vs. guessing the average":
                f"{nat(ridge.loc[s, 'mae_natural'])} vs. {nat(mean_only.loc[s, 'mae_natural'])}",
            "Practical threshold": threshold_text(s),
            "Sensitivity checks passed": f"{int(sens.loc[s, 'sum'])} of {int(sens.loc[s, 'size'])}" if s in sens.index else "",
            "Verdict": "Predictable" if prim_go.get(s, False) else "Not predictable",
        })
    pred_table = pd.DataFrame(rows_out)
    display(pred_table.style.hide(axis="index"))
    pred_table.to_csv(FIG_DIR / "11_predictability_table.csv", index=False)
    print(f"Saved {FIG_DIR / '11_predictability_table.csv'}")
    print("Sensitivity checks: precise dams weighted more, complete cases only, removal year added, weak-support "
          "dams left out, years 1–3 only, 30 km window (about twice the dams).")

**How to say it on a slide**
- **Dam removals don't all have the same effect.** About half of the dam-to-dam spread is real, not noise.
- **But what we know before a removal doesn't predict which dams will change most.** For dams in regions the model
  hadn't seen, it explained about 0–2% of the differences, below the 5% bar for all six measures. A flexible
  model (random forest) did no better, and neither did knowing the dam's region.
- **The result holds up.** It doesn't change with weighting, dropping uncertain dams, years 1–3 only, or a 30 km
  window with about twice as many dams.
- **What this means:** the average effects in sections 1–4 are the reliable guide. Predicting a specific dam would
  need site details we don't have nationally, such as stored sediment volume, how the dam was removed, and
  channel form.
- **Careful wording:** "not predictable *from these characteristics with this data*" (about 70–130 dams per
  measure, 6–8 regions). Don't say "dam traits don't matter".

## Other figures worth reusing

- **Map of the dams** (interactive, red = BACI, blue = before/after only): `distance_model_results.ipynb`,
  "Where are the dams?". Take a screenshot zoomed to the lower 48.
- **Verdict grid by distance**: `distance_model_results.ipynb` section 2 (second figure).